In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import tree
from sklearn import ensemble
import xgboost
np.random.seed(7)

In [4]:
data = pd.read_excel("career_dataset_large.xlsx")

In [5]:
data.head()

,Education Level,Specialization,Skills,Certifications,CGPA/Percentage,Recommended Career
0,Bachelor's,Finance,"Counseling, MS Office, Machine Learning",Tally ERP,67,Business Analyst
1,Intermediate,Science,"Accounting, MS Office",AWS Certified,67,Software Engineer
2,Master's,Business,"Accounting, SQL, Data Analysis",Mental Health Basics,90,Financial Analyst
3,Bachelor's,Computer Science,Communication,NaN,75,Clerk
4,Matric,Business,Data Analysis,Tally ERP,83,Sales Assistant


In [6]:
data.describe()

,CGPA/Percentage
count,5000.000000
mean,77.479800
std,10.396288
min,60.000000
25%,68.000000
50%,77.000000
75%,87.000000
max,95.000000


In [7]:
columns_to_binarize = [ 0, 1, 2, 3]
binary_to_index = {}

for column_index in columns_to_binarize:
    for value in data.iloc[:, column_index]:
        if pd.notna(value):
            for splits in str(value).split(', '):
                binary_to_index[splits] = column_index
binary_to_index

{"Bachelor's": 0,
 'Intermediate': 0,
 "Master's": 0,
 'Matric': 0,
 'PhD': 0,
 'Finance': 1,
 'Science': 1,
 'Business': 1,
 'Computer Science': 1,
 'Arts': 1,
 'Psychology': 1,
 'Commerce': 1,
 'Engineering': 1,
 'Counseling': 2,
 'MS Office': 2,
 'Machine Learning': 2,
 'Accounting': 2,
 'SQL': 2,
 'Data Analysis': 2,
 'Communication': 2,
 'Financial Analysis': 2,
 'Python': 2,
 'Marketing': 2,
 'Tally ERP': 3,
 'AWS Certified': 3,
 'Mental Health Basics': 3,
 'Digital Marketing': 3,
 'CFA Level 1': 3,
 'Creative Writing': 3,
 'Google Data Analytics': 3}

In [8]:
feature_names = list(binary_to_index.keys()) + ['CGPA/Percentage']
feature_names

["Bachelor's",
 'Intermediate',
 "Master's",
 'Matric',
 'PhD',
 'Finance',
 'Science',
 'Business',
 'Computer Science',
 'Arts',
 'Psychology',
 'Commerce',
 'Engineering',
 'Counseling',
 'MS Office',
 'Machine Learning',
 'Accounting',
 'SQL',
 'Data Analysis',
 'Communication',
 'Financial Analysis',
 'Python',
 'Marketing',
 'Tally ERP',
 'AWS Certified',
 'Mental Health Basics',
 'Digital Marketing',
 'CFA Level 1',
 'Creative Writing',
 'Google Data Analytics',
 'CGPA/Percentage']

In [9]:
target = 'Recommended Career'

In [10]:
x = pd.DataFrame(np.zeros( (len(data), len(feature_names)), dtype=int), columns = feature_names)

In [11]:
for index in range(len(feature_names)):
    feature_name = feature_names[index]
    if feature_name not in binary_to_index:
        x[feature_name] = data[feature_name]
        continue
        
    column_index = binary_to_index[feature_name]

    for row in range(len(data)):
        value = data.iloc[row, column_index]
        if pd.notna(value):
            split = value.split(', ')
            x.iloc[row, index] = int(feature_name in split)

x

,Bachelor's,Intermediate,Master's,Matric,PhD,Finance,Science,Business,Computer Science,Arts,...,Python,Marketing,Tally ERP,AWS Certified,Mental Health Basics,Digital Marketing,CFA Level 1,Creative Writing,Google Data Analytics,CGPA/Percentage
0,1,0,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,67
1,0,1,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,67
2,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,90
3,1,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,75
4,0,0,0,1,0,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,0,0,0,1,0,0,0,0,0,0,...,1,0,1,0,0,0,0,0,0,90
4996,0,0,1,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,1,0,92
4997,0,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,63
4998,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,83


In [12]:
target_names = data[target].unique()

y = data[target]

label_encoder = preprocessing.LabelEncoder()
y_encoded = label_encoder.fit_transform( data[target] )

y_encoded

array([ 0, 11,  3, ...,  3, 11,  1], shape=(5000,))

In [99]:
x_trainval, x_test, y_trainval, y_test = model_selection.train_test_split(x, y_encoded, test_size=0.33, stratify=y_encoded, random_state = 7)

In [100]:
x_train, x_val, y_train, y_val = model_selection.train_test_split(x_trainval, y_trainval, test_size=0.2, stratify=y_trainval, random_state = 7)
scaler = preprocessing.StandardScaler()
scaler.fit(x_train)
x_train = scaler.transform(x_train)
x_val = scaler.transform(x_val)

In [109]:
scaler = preprocessing.StandardScaler()
scaler.fit(x_trainval)
x_trainval = scaler.transform(x_trainval)
x_test = scaler.transform(x_test)

In [104]:
def train_and_evaluate(model, x_train, y_train, x_val, y_val):
    model.fit(x_train, y_train)

    return model.score(x_val, y_val)

In [105]:
number_of_estimators = np.arange(10, 110, 10)
number_of_estimators

array([ 10,  20,  30,  40,  50,  60,  70,  80,  90, 100])

In [106]:
max_depths = np.arange(5, 55, 5)
max_depths

array([ 5, 10, 15, 20, 25, 30, 35, 40, 45, 50])

In [107]:
optimal_number_of_estimators = number_of_estimators[0]
optimal_depth = max_depths[0]
optimal_accuracy = 0

for num in number_of_estimators:
    for depth in max_depths:
        model = ensemble.RandomForestClassifier(num, max_depth=depth, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_val, y_val)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_number_of_estimators = num
            optimal_depth = depth

print(optimal_number_of_estimators, optimal_depth, optimal_accuracy)

30 30 0.10895522388059702


In [108]:
number_of_estimators = np.arange(optimal_number_of_estimators-9, optimal_number_of_estimators+11, 2)
number_of_estimators

max_depths = np.arange(optimal_depth-4, optimal_depth+6, 1)
max_depths

for num in number_of_estimators:
    for depth in max_depths:
        model = ensemble.RandomForestClassifier(num, max_depth=depth, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_val, y_val)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_number_of_estimators = num
            optimal_depth = depth

print(optimal_number_of_estimators, optimal_depth, optimal_accuracy)

35 26 0.11343283582089553


In [128]:
model = ensemble.RandomForestClassifier(optimal_number_of_estimators, max_depth=optimal_depth, random_state=7)

In [129]:
model.fit(x_trainval, y_trainval)

,n_estimators,np.int64(35)
,criterion,'gini'
,max_depth,np.int64(26)
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [130]:
y_test_predicted = model.predict(x_test)
y_test_predicted

array([10,  1,  0, ...,  0,  3, 10], shape=(1650,))

In [131]:
y_test_predicted_strings = label_encoder.inverse_transform(y_test_predicted)
y_test_predicted_strings

array(['School Counselor', 'Clerk', 'Business Analyst', ...,
       'Business Analyst', 'Financial Analyst', 'School Counselor'],
      shape=(1650,), dtype=object)

In [134]:
model.score(x_trainval, y_trainval)

0.9982089552238806

In [135]:
model.score(x_test, y_test)

0.08242424242424243

In [136]:
metrics.confusion_matrix(y_test, y_test_predicted)

array([[12, 15, 10, 13, 10, 12, 11,  8,  9, 12, 11,  7],
       [13, 16,  7, 15, 18, 11, 13, 11,  7,  8, 13, 11],
       [10, 22, 10, 10, 12, 16, 11,  7,  7,  9, 12, 10],
       [16, 15, 20, 11, 11,  8, 10,  9,  4,  9, 14,  9],
       [15, 10, 14, 16,  8, 11, 11,  7,  7, 11, 13, 13],
       [11, 14, 16, 10,  7, 14, 17, 12, 10, 11, 12,  8],
       [20, 11, 19,  9, 10, 15, 13,  8,  7, 17,  7,  7],
       [ 9, 24, 12,  3,  8,  9, 14,  9, 13, 14, 13,  6],
       [14, 16,  6, 13, 13, 18, 10,  8, 10,  7, 12, 11],
       [11, 12, 15, 10, 11,  7, 15, 10, 13, 12,  8, 12],
       [13, 14, 16, 14,  9, 10, 15, 11,  9,  6, 12,  9],
       [ 8, 19, 19,  4, 11, 13, 12, 11, 10, 10, 12,  9]])

In [137]:
print(metrics.classification_report(y_test, y_test_predicted))

              precision    recall  f1-score   support

           0       0.08      0.09      0.09       130
           1       0.09      0.11      0.10       143
           2       0.06      0.07      0.07       136
           3       0.09      0.08      0.08       136
           4       0.06      0.06      0.06       136
           5       0.10      0.10      0.10       142
           6       0.09      0.09      0.09       143
           7       0.08      0.07      0.07       134
           8       0.09      0.07      0.08       138
           9       0.10      0.09      0.09       136
          10       0.09      0.09      0.09       138
          11       0.08      0.07      0.07       138

    accuracy                           0.08      1650
   macro avg       0.08      0.08      0.08      1650
weighted avg       0.08      0.08      0.08      1650



In [139]:
number_of_estimators = np.arange(50, 150, 10)

max_depths = np.arange(5, 25, 2)

optimal_number_of_estimators = number_of_estimators[0]
optimal_depth = max_depths[0]
optimal_accuracy = 0

for num in number_of_estimators:
    for depth in max_depths:
        model = ensemble.AdaBoostClassifier(estimator=tree.DecisionTreeClassifier(max_depth=depth), n_estimators=num, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_val, y_val)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_number_of_estimators = num
            optimal_depth = depth

print(optimal_number_of_estimators, optimal_depth, optimal_accuracy)

100 9 0.11791044776119403


In [140]:
model = ensemble.AdaBoostClassifier(estimator=tree.DecisionTreeClassifier(max_depth=optimal_depth), n_estimators=optimal_number_of_estimators, random_state=7)

In [141]:
model.fit(x_trainval, y_trainval)

,estimator,DecisionTreeC...h=np.int64(9))
,n_estimators,np.int64(100)
,learning_rate,1.0
,algorithm,'deprecated'
,random_state,7
,criterion,'gini'
,splitter,'best'
,max_depth,np.int64(9)
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0


In [142]:
y_test_predicted = model.predict(x_test)
y_test_predicted

array([4, 7, 8, ..., 0, 7, 7], shape=(1650,))

In [143]:
model.score(x_train, y_train)

0.9977611940298508

In [144]:
model.score(x_test, y_test)

0.07575757575757576

In [145]:
print(metrics.classification_report(y_test, y_test_predicted))

              precision    recall  f1-score   support

           0       0.07      0.06      0.07       130
           1       0.07      0.10      0.08       143
           2       0.05      0.06      0.06       136
           3       0.13      0.10      0.11       136
           4       0.08      0.06      0.07       136
           5       0.07      0.07      0.07       142
           6       0.10      0.10      0.10       143
           7       0.09      0.10      0.10       134
           8       0.07      0.08      0.07       138
           9       0.07      0.06      0.06       136
          10       0.06      0.07      0.06       138
          11       0.06      0.06      0.06       138

    accuracy                           0.08      1650
   macro avg       0.08      0.08      0.08      1650
weighted avg       0.08      0.08      0.08      1650



In [146]:
metrics.confusion_matrix(y_test, y_test_predicted)

array([[ 8, 17,  8,  6,  6, 11, 12, 13, 19,  8, 15,  7],
       [10, 14,  9, 10,  9, 11, 12, 17, 17, 11, 13, 10],
       [ 9, 17,  8, 10,  9, 18, 10,  8, 16,  9, 12, 10],
       [13, 10, 11, 14, 14, 13, 18,  6, 10,  9, 11,  7],
       [ 9, 18, 15,  7,  8, 11, 14, 11, 14,  8,  8, 13],
       [ 4, 22, 17,  8, 13, 10,  9, 13,  9,  9, 14, 14],
       [ 7, 19, 20,  4,  6, 11, 14, 14, 17,  9, 14,  8],
       [10, 16, 10,  4,  6, 13, 11, 13, 13, 18,  9, 11],
       [16, 22, 10, 14, 11, 13,  9,  4, 11,  6, 11, 11],
       [ 8, 10, 17,  8, 10, 13, 13,  9, 12,  8, 11, 17],
       [10, 17, 13, 10,  6, 13, 13, 11, 15,  9,  9, 12],
       [11, 15, 10, 14,  4, 12, 11, 20, 11,  8, 14,  8]])

In [147]:
number_of_estimators = np.arange(50, 150, 10)
number_of_estimators

max_depths = np.arange(5, 25, 2)
max_depths

optimal_number_of_estimators = number_of_estimators[0]
optimal_depth = max_depths[0]
optimal_accuracy = 0

for num in number_of_estimators:
    for depth in max_depths:
        model = xgboost.XGBClassifier(n_estimators=num, max_depth=depth, random_state=7)
        score = train_and_evaluate(model, x_train, y_train, x_val, y_val)
        
        if score > optimal_accuracy:
            optimal_accuracy = score
            optimal_number_of_estimators = num
            optimal_depth = depth

print(optimal_number_of_estimators, optimal_depth, optimal_accuracy)

50 19 0.11194029850746269


In [148]:
model = xgboost.XGBClassifier(n_estimators=optimal_number_of_estimators, max_depth=optimal_depth, random_state=7)

In [149]:
model.fit(x_trainval, y_trainval)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [150]:
y_test_predicted = model.predict(x_test)
y_test_predicted

array([ 5,  7, 11, ...,  2,  2,  7], shape=(1650,))

In [151]:
model.score(x_train, y_train)

0.2171641791044776

In [152]:
model.score(x_test, y_test)

0.07696969696969697

In [153]:
print(metrics.classification_report(y_test, y_test_predicted))

              precision    recall  f1-score   support

           0       0.08      0.07      0.08       130
           1       0.09      0.10      0.10       143
           2       0.05      0.07      0.06       136
           3       0.09      0.09      0.09       136
           4       0.09      0.07      0.08       136
           5       0.09      0.10      0.10       142
           6       0.08      0.08      0.08       143
           7       0.07      0.07      0.07       134
           8       0.06      0.05      0.05       138
           9       0.06      0.05      0.05       136
          10       0.07      0.07      0.07       138
          11       0.09      0.09      0.09       138

    accuracy                           0.08      1650
   macro avg       0.08      0.08      0.08      1650
weighted avg       0.08      0.08      0.08      1650



In [154]:
metrics.confusion_matrix(y_test, y_test_predicted)

array([[ 9, 12, 13, 13,  9, 10, 13, 10, 13,  6, 14,  8],
       [ 8, 15, 14, 14,  8,  9, 10, 19, 13, 11, 12, 10],
       [ 9, 16, 10,  9, 11, 17, 14,  6, 11, 10, 14,  9],
       [10, 11, 15, 12, 12, 10, 16,  9,  6, 10, 15, 10],
       [ 9, 10, 24, 18, 10, 10, 10,  9, 10,  8, 11,  7],
       [ 6, 14, 13,  9, 11, 14,  8, 14,  8, 12, 15, 18],
       [11, 14, 19, 10,  9, 12, 12,  8, 10, 14, 14, 10],
       [ 7, 16, 15,  5,  9, 15, 13,  9,  7, 17,  8, 13],
       [13, 14, 13, 16,  9, 19,  9,  9,  7, 11,  6, 12],
       [ 8, 11, 18,  9,  7, 10, 15,  7, 14,  7, 13, 17],
       [12, 13, 17,  8,  5,  6, 14, 16, 14, 12, 10, 11],
       [ 6, 12, 12, 12, 12, 17,  9, 12, 12,  9, 13, 12]])